In [1]:
import os
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [2]:
in_dir = '../../Data/score_gene_list'
out_dir = '../../results/08_gene_list/01_viz_gene_score'
data_dir = '../../results/06_spahdmap/02_prepare_data'

os.makedirs(out_dir, exist_ok=True)

In [3]:
sample_list = ['15837', '15852', '16512', '21022', '8262', 
               'B23-03794-5', 'B23-05785-7-1', 'B23-05785-7-2', 
               'X1842', 'X1845', 'X1899', 'X1912']
sample_list = ['X1912']

In [4]:
for sample in sample_list:
    os.makedirs(f"{out_dir}/{sample}", exist_ok=True)

    adata = sc.read_h5ad(os.path.join(data_dir, f'{sample}.h5ad'))
    adata.layers['count'] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    for file in os.listdir(in_dir):
        if file.endswith('.txt'):
            gene_score_df = pd.read_csv(os.path.join(in_dir, file), 
                                        index_col=0, sep='\t')
            
            gene_list = gene_score_df.index.tolist()
            # remove * from gene names
            gene_list = [gene.replace('*', '') for gene in gene_list]

            # convert gene names to uppercase
            gene_list = [gene.upper() for gene in gene_list]

            # overlap with adata var_names
            gene_list = list(set(gene_list) & set(adata.var_names))

            print(f"Processing {file} for sample {sample}: {len(gene_list)} genes found in adata")

            gene_set = file.split(".")[0]
            sc.tl.score_genes(adata, 
                            gene_list=gene_list, 
                            score_name=gene_set)

            fig, ax = plt.subplots(figsize=(6, 6))
            sc.pl.spatial(adata, color=gene_set, show=False, ax=ax)

            fig.savefig(f"{out_dir}/{sample}/{gene_set}.png", dpi=300)
            plt.close(fig)

    adata.write_h5ad(f'{out_dir}/{sample}/adata.h5ad')  

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Processing ECM_COLLAGENS_genes.txt for sample X1912: 44 genes found in adata
Processing ECM_PROTEOGLYCANS_genes.txt for sample X1912: 20 genes found in adata
Processing IFN_genes_all.txt for sample X1912: 214 genes found in adata
Processing HALLMARK_INTERFERON_ALPHA_RESPONSE.txt for sample X1912: 93 genes found in adata
Processing HALLMARK_INTERFERON_GAMMA_RESPONSE.txt for sample X1912: 188 genes found in adata
Processing Complement_score_genes.txt for sample X1912: 37 genes found in adata
Processing ECM_GLYCOPROTEINS_genes.txt for sample X1912: 121 genes found in adata
